# Task: Update SON Nature Layers for Water Quality V2.2
- Use McDowell et al (2025) at HydroBasin L6 where available
data = https://figshare.com/articles/dataset/_b_Anthropogenic_nutrient_inputs_cause_excessive_algal_growth_for_nearly_half_the_world_s_population_b_/24188787/1
- All_contams_DF_L6_Preds_v3: geopackage containing all GIS data.
- Used "TN_curr_v3" to replace "tnc_raw" values from 2020 data (these represented the median concentration)
- Used "TP_curr_v3" to replace "tpc_raw" values from 2020 data (these represented the median concentration)
- Fill in gaps with McDowell et al (2020) (see visual here, data = https://figshare.com/s/e44fe7319585183d888f) 

Include fields to indicate to users which basins:
- Use McDowell et al (2020) or (2025)
- Reference concentrations exceed thresholds*

Previous GitHub Repo: https://github.com/rafaexx/sbtn-SoN-water/blob/main/sbtn-SoN-water.qmd 

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

In [2]:
#SBTN File
ogson = gpd.read_file('sbnt_son_water_v2.1/sbtn_son_water_v2.1.shp', encoding='utf-8-sig')
ogson

,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500..."
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554..."
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369..."
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),2.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909..."
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),2.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,2.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),1.704707,0.140807,12.106649,P-limited,4.0,P-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633..."
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),1.727281,0.138922,12.433442,P-limited,4.0,P-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061..."
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),3.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,P-limited,3.0,P-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304..."
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.000000,1.0,Low (<5%),3.0,-0.617357,2.0,Low - Medium (-5 to 0),0.435813,2.0,Low Concentration (>0.4-0.8),0.835553,0.070579,11.838471,P-limited,3.0,P-limited growth undesirable,"POLYGON ((90.66250 53.77917, 90.66029 53.77973..."


In [3]:
#2025 data
df = gpd.read_file('06_Bootstrap_DF_L6/All_contams_DF_L6_Preds_bootstrap4.shp')
#remove rows with no hybasid
# df.dropna(subset=['HYBAS_ID_t'], inplace=True)
#add human enrichment 
df['TP_human'] = df['TP_curr_v3'] - df['TP_zero_v3'] 
df['TN_human']= df['TN_curr_v3'] - df['TN_zero_v3'] 

#make human zero if negative
df.loc[df["TP_human"] < 0, "TP_human"] = 0
df.loc[df["TN_human"] < 0, "TN_human"] = 0

df

,HYBAS_ID_t,TN_curr_v3,TN_zero_v3,TP_curr_v3,TP_zero_v3,NO3_curr_v,NO3_zero_v,DRP_curr_v,DRP_zero_v,Cases_curr,Cases_zero,TN_Diff_v3,TP_Diff_v3,NO3_Diff_v,DRP_Diff_v,geometry,TP_human,TN_human
0,3060189170,0.280350,0.279503,0.018888,0.019283,0.039869,0.050766,0.008802,0.009490,3,3,0.302010,-2.090727,-27.332953,-7.817991,"POLYGON ((66.89167 66.21250, 66.86758 66.21191...",0.000000,0.000847
1,3060194170,0.234266,0.274007,0.025001,0.023443,0.040230,0.049034,0.009591,0.011215,3,3,-16.964172,6.233854,-21.884271,-16.931847,"POLYGON ((66.32500 66.42917, 66.32465 66.38508...",0.001559,0.000000
2,3060194280,0.278985,0.278162,0.019055,0.019748,0.036754,0.047488,0.010282,0.012065,3,3,0.294813,-3.634347,-29.204784,-17.340914,"POLYGON ((67.88750 65.96667, 67.92083 65.96667...",0.000000,0.000822
3,3060198760,0.234352,0.274009,0.025006,0.023352,0.039605,0.047662,0.009842,0.011639,3,3,-16.921847,6.613178,-20.341926,-18.262783,"MULTIPOLYGON (((66.18750 66.22917, 66.18871 66...",0.001654,0.000000
4,3060203170,0.245818,0.278302,0.026193,0.024393,0.033353,0.042474,0.010326,0.011091,3,3,-13.214235,6.872488,-27.347613,-7.405258,"POLYGON ((64.18333 65.47083, 64.14341 65.47119...",0.001800,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11068,6060043520,0.560905,0.560998,0.048757,0.049578,0.052920,0.053691,0.010476,0.006965,3,3,-0.016579,-1.684551,-1.456689,33.516209,"POLYGON ((-67.38333 -47.87083, -67.38277 -47.8...",0.000000,0.000000
11069,6060043660,0.398700,0.386803,0.047190,0.047374,0.056927,0.057818,0.019743,0.019095,3,3,2.984109,-0.388906,-1.564783,3.279671,"POLYGON ((-63.92083 -34.68333, -63.92148 -34.6...",0.000000,0.011898
11070,6060043770,0.554133,0.552303,0.047679,0.048414,0.053633,0.054126,0.010783,0.007728,3,5,0.330374,-1.541471,-0.918815,28.330277,"POLYGON ((-66.12500 -47.63333, -66.12403 -47.6...",0.000000,0.001831
11071,6060044100,0.507555,0.393871,0.063229,0.061206,0.106815,0.101210,0.010544,0.007791,3,1,22.398408,3.199894,5.247448,26.109788,"POLYGON ((-60.93333 -35.52500, -60.92580 -35.5...",0.002023,0.113684


In [4]:
# #rename to match previous McDowell columns
df = df.rename(columns={"TN_curr_v3": "tnc_raw", "TP_curr_v3": "tpc_raw"})
df

,HYBAS_ID_t,tnc_raw,TN_zero_v3,tpc_raw,TP_zero_v3,NO3_curr_v,NO3_zero_v,DRP_curr_v,DRP_zero_v,Cases_curr,Cases_zero,TN_Diff_v3,TP_Diff_v3,NO3_Diff_v,DRP_Diff_v,geometry,TP_human,TN_human
0,3060189170,0.280350,0.279503,0.018888,0.019283,0.039869,0.050766,0.008802,0.009490,3,3,0.302010,-2.090727,-27.332953,-7.817991,"POLYGON ((66.89167 66.21250, 66.86758 66.21191...",0.000000,0.000847
1,3060194170,0.234266,0.274007,0.025001,0.023443,0.040230,0.049034,0.009591,0.011215,3,3,-16.964172,6.233854,-21.884271,-16.931847,"POLYGON ((66.32500 66.42917, 66.32465 66.38508...",0.001559,0.000000
2,3060194280,0.278985,0.278162,0.019055,0.019748,0.036754,0.047488,0.010282,0.012065,3,3,0.294813,-3.634347,-29.204784,-17.340914,"POLYGON ((67.88750 65.96667, 67.92083 65.96667...",0.000000,0.000822
3,3060198760,0.234352,0.274009,0.025006,0.023352,0.039605,0.047662,0.009842,0.011639,3,3,-16.921847,6.613178,-20.341926,-18.262783,"MULTIPOLYGON (((66.18750 66.22917, 66.18871 66...",0.001654,0.000000
4,3060203170,0.245818,0.278302,0.026193,0.024393,0.033353,0.042474,0.010326,0.011091,3,3,-13.214235,6.872488,-27.347613,-7.405258,"POLYGON ((64.18333 65.47083, 64.14341 65.47119...",0.001800,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11068,6060043520,0.560905,0.560998,0.048757,0.049578,0.052920,0.053691,0.010476,0.006965,3,3,-0.016579,-1.684551,-1.456689,33.516209,"POLYGON ((-67.38333 -47.87083, -67.38277 -47.8...",0.000000,0.000000
11069,6060043660,0.398700,0.386803,0.047190,0.047374,0.056927,0.057818,0.019743,0.019095,3,3,2.984109,-0.388906,-1.564783,3.279671,"POLYGON ((-63.92083 -34.68333, -63.92148 -34.6...",0.000000,0.011898
11070,6060043770,0.554133,0.552303,0.047679,0.048414,0.053633,0.054126,0.010783,0.007728,3,5,0.330374,-1.541471,-0.918815,28.330277,"POLYGON ((-66.12500 -47.63333, -66.12403 -47.6...",0.000000,0.001831
11071,6060044100,0.507555,0.393871,0.063229,0.061206,0.106815,0.101210,0.010544,0.007791,3,1,22.398408,3.199894,5.247448,26.109788,"POLYGON ((-60.93333 -35.52500, -60.92580 -35.5...",0.002023,0.113684


In [5]:
df = df[['HYBAS_ID_t','tpc_raw', 'tnc_raw','TN_human', 'TP_human']]
df['ratio_n_p'] = df['tnc_raw']/df['tpc_raw']
cond = [
    df["ratio_n_p"] >= 15,
    (df["ratio_n_p"] >= 7) & (df["ratio_n_p"] < 15),
    df["ratio_n_p"] < 7]
values=["P-limited",'Co-limited',"N-limited"]
df['limiting']=np.select(cond, values, default=None)
df

,HYBAS_ID_t,tpc_raw,tnc_raw,TN_human,TP_human,ratio_n_p,limiting
0,3060189170,0.018888,0.280350,0.000847,0.000000,14.842402,Co-limited
1,3060194170,0.025001,0.234266,0.000000,0.001559,9.370154,Co-limited
2,3060194280,0.019055,0.278985,0.000822,0.000000,14.640996,Co-limited
3,3060198760,0.025006,0.234352,0.000000,0.001654,9.371853,Co-limited
4,3060203170,0.026193,0.245818,0.000000,0.001800,9.384856,Co-limited
...,...,...,...,...,...,...,...
11068,6060043520,0.048757,0.560905,0.000000,0.000000,11.504108,Co-limited
11069,6060043660,0.047190,0.398700,0.011898,0.000000,8.448784,Co-limited
11070,6060043770,0.047679,0.554133,0.001831,0.000000,11.622247,Co-limited
11071,6060044100,0.063229,0.507555,0.113684,0.002023,8.027217,Co-limited


In [6]:
# Five-class screening bands based on the McDowell et al. (2025)
# ecological thresholds. Lower subdivisions are screening bands rather
# than independently validated ecological response thresholds.
conditions = [
    # N-limited: equality is assigned to the higher-risk band
    (df["limiting"] == "N-limited") & (df["tnc_raw"] < 0.2),
    (df["limiting"] == "N-limited") & (df["tnc_raw"] >= 0.2) & (df["tnc_raw"] < 0.4),
    (df["limiting"] == "N-limited") & (df["tnc_raw"] >= 0.4) & (df["tnc_raw"] < 0.8),
    (df["limiting"] == "N-limited") & (df["tnc_raw"] >= 0.8) & (df["tnc_raw"] < 1.35),
    (df["limiting"] == "N-limited") & (df["tnc_raw"] >= 1.35),

    # P-limited: equality is assigned to the higher-risk band
    (df["limiting"] == "P-limited") & (df["tpc_raw"] < 0.01),
    (df["limiting"] == "P-limited") & (df["tpc_raw"] >= 0.01) & (df["tpc_raw"] < 0.023),
    (df["limiting"] == "P-limited") & (df["tpc_raw"] >= 0.023) & (df["tpc_raw"] < 0.046),
    (df["limiting"] == "P-limited") & (df["tpc_raw"] >= 0.046) & (df["tpc_raw"] < 0.078),
    (df["limiting"] == "P-limited") & (df["tpc_raw"] >= 0.078),

    # Co-limited: the lower of the N and P screening bands controls.
    # Growth becomes undesirable only when both N >= 0.800 and P >= 0.046.
    (df["limiting"] == "Co-limited") & ((df["tnc_raw"] < 0.2) | (df["tpc_raw"] < 0.010)),
    (df["limiting"] == "Co-limited") & ((df["tnc_raw"] < 0.4) | (df["tpc_raw"] < 0.023)),
    (df["limiting"] == "Co-limited") & ((df["tnc_raw"] < 0.8) | (df["tpc_raw"] < 0.046)),
    (df["limiting"] == "Co-limited") & ((df["tnc_raw"] < 1.35) | (df["tpc_raw"] < 0.078)),
    (df["limiting"] == "Co-limited"),
]

choices = [1, 2, 3, 4, 5,
           1, 2, 3, 4, 5,
           1, 2, 3, 4, 5]

df["pgp_n"] = np.select(conditions, choices, default=np.nan)
df

,HYBAS_ID_t,tpc_raw,tnc_raw,TN_human,TP_human,ratio_n_p,limiting,pgp_n
0,3060189170,0.018888,0.280350,0.000847,0.000000,14.842402,Co-limited,2.0
1,3060194170,0.025001,0.234266,0.000000,0.001559,9.370154,Co-limited,2.0
2,3060194280,0.019055,0.278985,0.000822,0.000000,14.640996,Co-limited,2.0
3,3060198760,0.025006,0.234352,0.000000,0.001654,9.371853,Co-limited,2.0
4,3060203170,0.026193,0.245818,0.000000,0.001800,9.384856,Co-limited,2.0
...,...,...,...,...,...,...,...,...
11068,6060043520,0.048757,0.560905,0.000000,0.000000,11.504108,Co-limited,3.0
11069,6060043660,0.047190,0.398700,0.011898,0.000000,8.448784,Co-limited,2.0
11070,6060043770,0.047679,0.554133,0.001831,0.000000,11.622247,Co-limited,3.0
11071,6060044100,0.063229,0.507555,0.113684,0.002023,8.027217,Co-limited,3.0


In [7]:
conditions = [
    (df['limiting'] == "N-limited") & (df['pgp_n'] == 1),
    (df['limiting'] == "N-limited") & (df['pgp_n'] == 2),
    (df['limiting'] == "N-limited") & (df['pgp_n'] == 3),
    (df['limiting'] == "N-limited") & (df['pgp_n'] == 4),
    (df['limiting'] == "N-limited") & (df['pgp_n'] == 5),

    (df['limiting'] == "P-limited") & (df['pgp_n'] == 1),
    (df['limiting'] == "P-limited") & (df['pgp_n'] == 2),
    (df['limiting'] == "P-limited") & (df['pgp_n'] == 3),
    (df['limiting'] == "P-limited") & (df['pgp_n'] == 4),
    (df['limiting'] == "P-limited") & (df['pgp_n'] == 5),

    # McDowell co-limited criteria
    (df['limiting'] == "Co-limited") & ((df["tnc_raw"] < 0.8) | (df["tpc_raw"] < 0.046)),
    (df['limiting'] == "Co-limited") & (df["tnc_raw"] >= 0.8) & (df["tpc_raw"] >= 0.046),
]

choices = [
    "N-limited growth acceptable",
    "N-limited growth acceptable",
    "N-limited growth acceptable",
    "N-limited growth undesirable",
    "N-limited growth undesirable",

    "P-limited growth acceptable",
    "P-limited growth acceptable",
    "P-limited growth acceptable",
    "P-limited growth undesirable",
    "P-limited growth undesirable",

    "Co-limited growth acceptable",
    "Co-limited growth undesirable",
]

df["pgp_label"] = np.select(conditions, choices, default=None)
df

,HYBAS_ID_t,tpc_raw,tnc_raw,TN_human,TP_human,ratio_n_p,limiting,pgp_n,pgp_label
0,3060189170,0.018888,0.280350,0.000847,0.000000,14.842402,Co-limited,2.0,Co-limited growth acceptable
1,3060194170,0.025001,0.234266,0.000000,0.001559,9.370154,Co-limited,2.0,Co-limited growth acceptable
2,3060194280,0.019055,0.278985,0.000822,0.000000,14.640996,Co-limited,2.0,Co-limited growth acceptable
3,3060198760,0.025006,0.234352,0.000000,0.001654,9.371853,Co-limited,2.0,Co-limited growth acceptable
4,3060203170,0.026193,0.245818,0.000000,0.001800,9.384856,Co-limited,2.0,Co-limited growth acceptable
...,...,...,...,...,...,...,...,...,...
11068,6060043520,0.048757,0.560905,0.000000,0.000000,11.504108,Co-limited,3.0,Co-limited growth acceptable
11069,6060043660,0.047190,0.398700,0.011898,0.000000,8.448784,Co-limited,2.0,Co-limited growth acceptable
11070,6060043770,0.047679,0.554133,0.001831,0.000000,11.622247,Co-limited,3.0,Co-limited growth acceptable
11071,6060044100,0.063229,0.507555,0.113684,0.002023,8.027217,Co-limited,3.0,Co-limited growth acceptable


## Join the New McDowell to the old and create column to indicate which will be used

In [8]:
# add '2025_' to all column names
df = df.add_prefix('2025_')
df

,2025_HYBAS_ID_t,2025_tpc_raw,2025_tnc_raw,2025_TN_human,2025_TP_human,2025_ratio_n_p,2025_limiting,2025_pgp_n,2025_pgp_label
0,3060189170,0.018888,0.280350,0.000847,0.000000,14.842402,Co-limited,2.0,Co-limited growth acceptable
1,3060194170,0.025001,0.234266,0.000000,0.001559,9.370154,Co-limited,2.0,Co-limited growth acceptable
2,3060194280,0.019055,0.278985,0.000822,0.000000,14.640996,Co-limited,2.0,Co-limited growth acceptable
3,3060198760,0.025006,0.234352,0.000000,0.001654,9.371853,Co-limited,2.0,Co-limited growth acceptable
4,3060203170,0.026193,0.245818,0.000000,0.001800,9.384856,Co-limited,2.0,Co-limited growth acceptable
...,...,...,...,...,...,...,...,...,...
11068,6060043520,0.048757,0.560905,0.000000,0.000000,11.504108,Co-limited,3.0,Co-limited growth acceptable
11069,6060043660,0.047190,0.398700,0.011898,0.000000,8.448784,Co-limited,2.0,Co-limited growth acceptable
11070,6060043770,0.047679,0.554133,0.001831,0.000000,11.622247,Co-limited,3.0,Co-limited growth acceptable
11071,6060044100,0.063229,0.507555,0.113684,0.002023,8.027217,Co-limited,3.0,Co-limited growth acceptable


In [9]:
ogson['HYBAS_ID'] = ogson['HYBAS_ID'].astype(int)
df['2025_HYBAS_ID_t'] = df['2025_HYBAS_ID_t'].astype(int)

df = ogson.merge(df, left_on = 'HYBAS_ID' , right_on =  '2025_HYBAS_ID_t', how = 'left')
df


,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry,2025_HYBAS_ID_t,2025_tpc_raw,2025_tnc_raw,2025_TN_human,2025_TP_human,2025_ratio_n_p,2025_limiting,2025_pgp_n,2025_pgp_label
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),2.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),2.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,2.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),1.704707,0.140807,12.106649,P-limited,4.0,P-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633...",3.060025e+09,0.081767,0.852877,0.271134,0.000466,10.430560,Co-limited,4.0,Co-limited growth undesirable
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),1.727281,0.138922,12.433442,P-limited,4.0,P-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061...",3.060026e+09,0.073468,0.832869,0.256965,0.000000,11.336473,Co-limited,4.0,Co-limited growth undesirable
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),3.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,P-limited,3.0,P-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.000000,1.0,Low (<5%),3.0,-0.617357,2.0,Low - Medium (-5 to 0),0.435813,2.0,Low Concentration (>0.4-0.8),0.835553,0.070579,11.838471,P-limited,3.0,P-limited growth undesir

In [10]:
# Identify rows with updated McDowell 2025 concentrations
has_2025_data = df['2025_tnc_raw'].notna() & df['2025_tpc_raw'].notna()
df['year_McD_data'] = np.where(has_2025_data, '2025', '2020')

# Preserve 2025 anthropogenic enrichment. The 2020 data do not contain
# modeled reference concentrations, so retain the previous assumption
# that the raw concentration represents the manageable concentration.
df['tn_human'] = np.where(has_2025_data, df['2025_TN_human'], df['tnc_raw'])
df['tp_human'] = np.where(has_2025_data, df['2025_TP_human'], df['tpc_raw'])

# Fill missing 2025 raw concentrations with the 2020 raw concentrations.
# Scores and labels are recalculated afterward using one methodology.
df['tnc_raw'] = df['2025_tnc_raw'].fillna(df['tnc_raw'])
df['tpc_raw'] = df['2025_tpc_raw'].fillna(df['tpc_raw'])
df

,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry,2025_HYBAS_ID_t,2025_tpc_raw,2025_tnc_raw,2025_TN_human,2025_TP_human,2025_ratio_n_p,2025_limiting,2025_pgp_n,2025_pgp_label,year_McD_data,tn_human,tp_human
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.743598,0.191573
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.749842,0.143022
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,2.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.463732,0.116853
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),2.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,2.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.790747,0.138488
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),2.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,2.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.757878,0.167163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),0.852877,0.081767,12.106649,P-limited,4.0,P-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633...",3.060025e+09,0.081767,0.852877,0.271134,0.000466,10.430560,Co-limited,4.0,Co-limited growth undesirable,2025,0.271134,0.000466
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),0.832869,0.073468,12.433442,P-limited,4.0,P-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061...",3.060026e+09,0.073468,0.832869,0.256965,0.000000,11.336473,Co-limited,4.0,Co-limited growth undesirable,2025,0.256965,0.000000
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),3.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,P-limited,3.0,P-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020,0.840846,0.092100
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Ar

In [11]:
# Recalculate the McDowell classification for the combined 2020/2025 data
df['ratio_n_p'] = df['tnc_raw'] / df['tpc_raw']

limiting_conditions = [
    df['ratio_n_p'] < 7,
    (df['ratio_n_p'] >= 7) & (df['ratio_n_p'] < 15),
    df['ratio_n_p'] >= 15
]
limiting_values = ['N-limited', 'Co-limited', 'P-limited']
df['limiting'] = np.select(limiting_conditions, limiting_values, default=None)

score_conditions = [
    # N-limited screening bands
    (df['limiting'] == 'N-limited') & (df['tnc_raw'] < 0.20),
    (df['limiting'] == 'N-limited') & (df['tnc_raw'] >= 0.20) & (df['tnc_raw'] < 0.40),
    (df['limiting'] == 'N-limited') & (df['tnc_raw'] >= 0.40) & (df['tnc_raw'] < 0.80),
    (df['limiting'] == 'N-limited') & (df['tnc_raw'] >= 0.80) & (df['tnc_raw'] < 1.35),
    (df['limiting'] == 'N-limited') & (df['tnc_raw'] >= 1.35),

    # P-limited screening bands
    (df['limiting'] == 'P-limited') & (df['tpc_raw'] < 0.010),
    (df['limiting'] == 'P-limited') & (df['tpc_raw'] >= 0.010) & (df['tpc_raw'] < 0.023),
    (df['limiting'] == 'P-limited') & (df['tpc_raw'] >= 0.023) & (df['tpc_raw'] < 0.046),
    (df['limiting'] == 'P-limited') & (df['tpc_raw'] >= 0.046) & (df['tpc_raw'] < 0.078),
    (df['limiting'] == 'P-limited') & (df['tpc_raw'] >= 0.078),

    # Co-limited: the lower of the N and P screening bands controls
    (df['limiting'] == 'Co-limited') & ((df['tnc_raw'] < 0.20) | (df['tpc_raw'] < 0.010)),
    (df['limiting'] == 'Co-limited') & ((df['tnc_raw'] < 0.40) | (df['tpc_raw'] < 0.023)),
    (df['limiting'] == 'Co-limited') & ((df['tnc_raw'] < 0.80) | (df['tpc_raw'] < 0.046)),
    (df['limiting'] == 'Co-limited') & ((df['tnc_raw'] < 1.35) | (df['tpc_raw'] < 0.078)),
    df['limiting'] == 'Co-limited'
]

score_values = [
    1, 2, 3, 4, 5,
    1, 2, 3, 4, 5,
    1, 2, 3, 4, 5
]
df['pgp_n'] = np.select(score_conditions, score_values, default=np.nan)

# Apply one consistent label rule to both source years
growth_status = np.where(df['pgp_n'] <= 3, 'acceptable', 'undesirable')
df['pgp_label'] = np.where(
    df['pgp_n'].isna(),
    None,
    df['limiting'] + ' growth ' + growth_status
)

# Remove temporary 2025-prefixed fields after recalculation
df = df.drop(columns=[column for column in df.columns if column.startswith('2025_')])
df


,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry,year_McD_data,tn_human,tp_human
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500...",2020,0.743598,0.191573
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554...",2020,0.749842,0.143022
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369...",2020,0.463732,0.116853
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),2.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909...",2020,0.790747,0.138488
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),2.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,3.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27...",2020,0.757878,0.167163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),0.852877,0.081767,10.430560,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633...",2025,0.271134,0.000466
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),0.832869,0.073468,11.336473,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061...",2025,0.256965,0.000000
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),3.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304...",2020,0.840846,0.092100
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.000000,1.0,Low (<5%),3.0,-0.617357,2.0,Low - Medium (-5 to 0),0.435813,2.0,Low Concentration (>0.4-0.8),0.268568,0.029739,9.030870,Co-limited,2.0,Co-limited growth acceptable,"POLYGON ((90.66250 53.77917, 90.66029 53.77973...",2025,0.010766,0.000649


In [12]:
#add in percent reductions
#replace with just human for 2025 but not for 2020 because they don't have this data 
df['p_req_reduction'] =(df['tpc_raw']-0.046)/df['tp_human'] 
df['n_req_reduction'] = (df['tnc_raw']-0.80)/df['tn_human']

# Set negative reductions to 0
df.loc[df["p_req_reduction"] < 0, "p_req_reduction"] = 0
df.loc[df["n_req_reduction"] < 0, "n_req_reduction"] = 0

# Set N reduction to 0 when N-limited growth is acceptable
df.loc[df["pgp_label"] == "N-limited growth acceptable", "n_req_reduction"] = 0
df.loc[df["pgp_label"] == "N-limited growth acceptable", "p_req_reduction"] = 0
df.loc[df["pgp_label"] == "N-limited growth undesirable", "p_req_reduction"] = 0
df.loc[df["pgp_label"] == "P-limited growth acceptable", "n_req_reduction"] = 0
df.loc[df["pgp_label"] == "P-limited growth acceptable", "p_req_reduction"] = 0
df.loc[df["pgp_label"] == "P-limited growth undesirable", "n_req_reduction"] = 0
df.loc[df["pgp_label"] == "Co-limited growth acceptable", "n_req_reduction"] = 0
df.loc[df["pgp_label"] == "Co-limited growth acceptable", "p_req_reduction"] = 0
df


,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry,year_McD_data,tn_human,tp_human,p_req_reduction,n_req_reduction
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500...",2020,0.743598,0.191573,0.000000,0.000000
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554...",2020,0.749842,0.143022,0.000000,0.000000
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369...",2020,0.463732,0.116853,0.000000,0.000000
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),2.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909...",2020,0.790747,0.138488,0.000000,0.000000
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),2.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,3.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27...",2020,0.757878,0.167163,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),0.852877,0.081767,10.430560,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633...",2025,0.271134,0.000466,76.722354,0.195022
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),0.832869,0.073468,11.336473,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061...",2025,0.256965,0.000000,inf,0.127911
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),3.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304...",2020,0.840846,0.092100,0.500545,0.048577
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.000000,1.0,Low (<5%),3.0,-0.617357,2.0,Low - Medium (-5 to 0),0.435813,2.0,Low Concentration (>0.4-0.8),0.268568,0.029739,9.030870,Co-limited,2.0,Co-limited growth acceptable,"POLYGON ((90.66250 53.77917, 90.66029 53.77973...",2025,0.010766,0.000649,0.000000,0.000000


In [13]:
#Update the wp_max
df['wp_max'] = df[["cep_n", "nox_n", "pgp_n"]].max(axis=1)
df = df.replace('–', '-')
df

,HYBAS_ID,WMOBB_id,WMOBB_name,region,wa_max,bws_raw,bws_n,bws_label,wdp_raw,wdp_n,wdp_label,fqt_raw,fqt_n,fqt_label,wp_max,cep_raw,cep_n,cep_label,nox_raw,nox_n,nox_label,tnc_raw,tpc_raw,ratio_n_p,limiting,pgp_n,pgp_label,geometry,year_McD_data,tn_human,tp_human,p_req_reduction,n_req_reduction
0,1060000010,176.0,Red Sea,Africa,5.0,NaN,5.0,Extremely High (>80%),4.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.959230,3.0,Moderate Concentration (>0.8-1.2),0.743598,0.191573,3.881547,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.36667 29.67917, 32.36667 29.67500...",2020,0.743598,0.191573,0.000000,0.000000
1,1060000100,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.026442,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.532705,2.0,Low Concentration (>0.4-0.8),0.749842,0.143022,5.242843,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.73333 29.88333, 31.73390 29.88554...",2020,0.749842,0.143022,0.000000,0.000000
2,1060000110,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,4.000000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,0.000000,3.0,Medium - High (0 to 1),0.528857,2.0,Low Concentration (>0.4-0.8),0.463732,0.116853,3.968518,N-limited,3.0,N-limited growth acceptable,"POLYGON ((32.38750 29.38333, 32.38508 29.38369...",2020,0.463732,0.116853,0.000000,0.000000
3,1060000150,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,2.500000,2.0,>5–25% annual depletion,0.998406,5.0,Extreme (>=80%),3.0,-0.015514,2.0,Low - Medium (-5 to 0),0.438657,2.0,Low Concentration (>0.4-0.8),0.790747,0.138488,5.709859,N-limited,3.0,N-limited growth acceptable,"POLYGON ((31.97500 29.16667, 31.97465 29.16909...",2020,0.790747,0.138488,0.000000,0.000000
4,1060000160,176.0,Red Sea,Africa,5.0,1.000000,5.0,Arid and Low Water Use,3.500000,4.0,">25–75% annual depletion, Seasonal",0.998406,5.0,Extreme (>=80%),3.0,-0.005207,2.0,Low - Medium (-5 to 0),0.238231,1.0,Very Low Concentration (<=0.4),0.757878,0.167163,4.533764,N-limited,3.0,N-limited growth acceptable,"MULTIPOLYGON (((33.67778 27.62917, 33.67119 27...",2020,0.757878,0.167163,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,3060025460,211.0,Irtysh,Asia,5.0,1.214962,5.0,Extremely High (>80%),2.000000,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.765273,2.0,Low Concentration (>0.4-0.8),0.852877,0.081767,10.430560,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((79.25833 52.85417, 79.25895 52.85633...",2025,0.271134,0.000466,76.722354,0.195022
16393,3060025570,211.0,Irtysh,Asia,2.0,0.022355,1.0,Low (<10%),2.246523,2.0,>5–25% annual depletion,0.000000,1.0,Low (<5%),4.0,-0.592593,2.0,Low - Medium (-5 to 0),0.499121,2.0,Low Concentration (>0.4-0.8),0.832869,0.073468,11.336473,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((75.19583 53.75000, 75.19366 53.75061...",2025,0.256965,0.000000,inf,0.127911
16394,3060026330,211.0,Irtysh,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.107156,2.0,Moderate (5-25%),4.0,0.172309,3.0,Medium - High (0 to 1),0.366522,1.0,Very Low Concentration (<=0.4),0.840846,0.092100,9.129664,Co-limited,4.0,Co-limited growth undesirable,"POLYGON ((70.65000 49.26250, 70.65057 49.27304...",2020,0.840846,0.092100,0.500545,0.048577
16395,3060026650,209.0,Yenisey,Asia,5.0,1.000000,5.0,Arid and Low Water Use,1.000000,1.0,<=5% annual depletion,0.000000,1.0,Low (<5%),2.0,-0.617357,2.0,Low - Medium (-5 to 0),0.435813,2.0,Low Concentration (>0.4-0.8),0.268568,0.029739,9.030870,Co-limited,2.0,Co-limited growth acceptable,"POLYGON ((90.66250 53.77917, 90.66029 53.77973...",2025,0.010766,0.000649,0.000000,0.000000


In [14]:
df.to_file('sbnt_son_water_v2.2/sbtn_son_water_v2.2.shp')

In [15]:
csv = df.drop(columns={'geometry'})
csv.to_csv('sbnt_son_water_v2.2/sbtn_son_water_v2.2.csv', encoding='utf-8-sig')

In [16]:
df.pgp_label.unique()

array(['N-limited growth acceptable', 'N-limited growth undesirable',
       'Co-limited growth acceptable', 'Co-limited growth undesirable',
       'P-limited growth acceptable', 'P-limited growth undesirable',
       None], dtype=object)